# Predicting Irrigation Need

Link to Competittion: https://www.kaggle.com/competitions/playground-series-s6e4/overview

## Imports

In [184]:
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['figure.figsize'] = (12, 6)

import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)

import xgboost as xgb

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, accuracy_score, confusion_matrix, mean_squared_error, classification_report, balanced_accuracy_score
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold, RandomizedSearchCV, train_test_split
from sklearn.cluster import KMeans

from common import *

In [185]:
from platform import python_version
print('python: ', python_version())
print('pandas: ', pd.__version__)
print('numpy: ', np.__version__)
import matplotlib
print('matplotlib: ', matplotlib.__version__)
print('seaborn: ', sns.__version__)
import sklearn
print('sklearn: ', sklearn.__version__)
print('xgboost: ', xgb.__version__)

python:  3.13.11
pandas:  2.3.3
numpy:  2.3.5
matplotlib:  3.10.8
seaborn:  0.13.2
sklearn:  1.8.0
xgboost:  3.2.0


## Helpers

## Load data

In [186]:
train_df = pd.read_csv('archive/train.csv')
test_df = pd.read_csv('archive/test.csv')

## Call the pipeline

In [187]:
df = (train_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

## Features

In [188]:
target = get_target()

In [189]:
features = get_features(df)

In [190]:
features

['soil_type',
 'soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm',
 'region']

In [191]:
categorical_features = [
    'crop_type',
    'crop_growth_stage',
    'season',
    'irrigation_type',
    'water_source',
    'soil_type',
    'region'
]

In [192]:
numerical_features = [f for f in features if f not in categorical_features]

In [193]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [194]:
numerical_features

['soil_ph',
 'soil_moisture',
 'organic_carbon',
 'electrical_conductivity',
 'temperature_c',
 'humidity',
 'rainfall_mm',
 'sunlight_hours',
 'wind_speed_kmh',
 'field_area_hectare',
 'mulching_used',
 'previous_irrigation_mm']

In [195]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [196]:
df[features]

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,0,112.16,East
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,1,47.16,South
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,1,110.38,North
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,1,53.85,South
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,0,93.19,South
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,0,118.36,South
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,1,40.75,Central
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,1,2.62,South
629998,Clay,5.93,42.86,0.33,3.39,34.99,94.58,2433.92,9.88,5.00,Sugarcane,Vegetative,Zaid,Sprinkler,Groundwater,0.79,1,90.00,East


In [197]:
df[target] = df[target].map({'Low': 0, 'Medium': 1, 'High': 2})

In [198]:
df[categorical_features] = df[categorical_features].astype('category')

In [199]:
categorical_features

['crop_type',
 'crop_growth_stage',
 'season',
 'irrigation_type',
 'water_source',
 'soil_type',
 'region']

In [200]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   soil_type                630000 non-null  category
 1   soil_ph                  630000 non-null  float64 
 2   soil_moisture            630000 non-null  float64 
 3   organic_carbon           630000 non-null  float64 
 4   electrical_conductivity  630000 non-null  float64 
 5   temperature_c            630000 non-null  float64 
 6   humidity                 630000 non-null  float64 
 7   rainfall_mm              630000 non-null  float64 
 8   sunlight_hours           630000 non-null  float64 
 9   wind_speed_kmh           630000 non-null  float64 
 10  crop_type                630000 non-null  category
 11  crop_growth_stage        630000 non-null  category
 12  season                   630000 non-null  category
 13  irrigation_type          630000 non-null  ca

In [201]:
X, y = df[features], df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y, random_state=123)

In [202]:
xgb_model = XGBClassifier(n_jobs=-1, random_state=123, enable_categorical=True)

In [203]:
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [204]:
xgb_model.score(X_test, y_test)

0.9847301587301587

In [205]:
balanced_accuracy_score(y_test, xgb_model.predict(X_test))

0.9611150993177486

In [206]:
xgb_final_model = XGBClassifier(n_jobs=-1, random_state=123, enable_categorical=True)
xgb_final_model.fit(df[features], df[target])

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [207]:
df = (test_df
          .pipe(copy_data)
          .pipe(clean_data)
          # .pipe(remove_outliers)
          .pipe(remove_duplicates)
          .pipe(make_new_features)
           )

In [208]:
df['mulching_used'] = df['mulching_used'].map({'Yes': 1, 'No': 0})

In [209]:
df[categorical_features] = df[categorical_features].astype('category')

In [210]:
df

,soil_type,soil_ph,soil_moisture,organic_carbon,electrical_conductivity,temperature_c,humidity,rainfall_mm,sunlight_hours,wind_speed_kmh,crop_type,crop_growth_stage,season,irrigation_type,water_source,field_area_hectare,mulching_used,previous_irrigation_mm,region
0,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,1,47.48,West
1,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,1,56.43,South
2,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,1,20.00,East
3,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,0,102.99,North
4,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,1,13.33,Central
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
269995,Sandy,5.63,51.90,0.68,2.58,33.27,72.09,2326.61,7.09,10.02,Potato,Vegetative,Rabi,Rainfed,River,2.93,1,43.49,East
269996,Loamy,7.84,45.16,0.85,1.04,27.55,45.16,2322.37,5.15,5.62,Wheat,Vegetative,Rabi,Canal,Groundwater,11.23,1,92.03,West
269997,Loamy,7.83,11.02,1.56,1.90,23.39,64.87,996.72,10.44,9.98,Maize,Vegetative,Zaid,Sprinkler,Groundwater,2.88,1,34.02,East
269998,Silt,7.12,10.18,1.32,2.65,41.09,58.04,1130.71,5.11,1.46,Rice,Harvest,Kharif,Rainfed,River,5.71,1,3.92,East


In [211]:
preds = xgb_final_model.predict(df[features])

In [212]:
preds

array([0, 0, 0, ..., 1, 0, 1], shape=(270000,))

In [213]:
submission = pd.DataFrame(test_df['id'].copy())

In [214]:
submission['Irrigation_Need'] = preds

In [215]:
submission

,id,Irrigation_Need
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0
...,...,...
269995,899995,1
269996,899996,0
269997,899997,1
269998,899998,0


In [216]:
submission['Irrigation_Need'] = submission['Irrigation_Need'].map({0: 'Low', 1: 'Medium', 2: 'High'})

In [217]:
submission['Irrigation_Need'].value_counts()

Irrigation_Need
Low       159841
Medium    101529
High        8630
Name: count, dtype: int64

In [218]:
submission.to_csv('archive/submission_02.csv', index=False, header=True)